# 🚀 Stage 5 — Graph Transformer for Blockchain Fraud Detection

**Goal:** Train a Graph Transformer (`TransformerConv`) on the Elliptic Bitcoin Dataset using a free Kaggle GPU (T4).

**Output:** `graph_transformer.pt` — download this and place it in your local `models/` directory.

**Pass condition:** Transformer Validation F1 exceeds GraphSAGE baseline (~0.85).

> ⚙️ **Kaggle Setup Checklist:**
> 1. Click **Settings** (three dots or right panel) → **Accelerator** → **GPU T4** (or T4 x2).
> 2. Click **Settings** → **Internet** → **Internet on** (required for `pip install`).
> 3. Ensure the Elliptic Bitcoin Dataset is added under **Data** / **Input**.

In [ ]:
# ============================================================
# CELL 1 — Install PyTorch Geometric & Check GPU
# ============================================================
import sys
import subprocess
import torch

print(f"Python version  : {sys.version.split()[0]}")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device      : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not detected! Please turn on GPU in Kaggle Settings (right sidebar).")

print("\n📦 Installing / Verifying PyTorch Geometric...")
try:
    import torch_geometric
    print(f"✅ torch-geometric {torch_geometric.__version__} is already installed!")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])
    import torch_geometric
    print(f"✅ torch-geometric {torch_geometric.__version__} installed successfully!")


In [ ]:
# ============================================================
# CELL 2 — Auto-Locate and Load Elliptic Dataset CSVs
# ============================================================
import os
from pathlib import Path
import numpy as np
import pandas as pd

# --- Auto-locate the 3 CSV files anywhere inside /kaggle/input ---
base_input = Path('/kaggle/input')

def find_csv_file(keywords):
    for root, _, files in os.walk(base_input):
        for f in files:
            f_lower = f.lower()
            if f_lower.endswith('.csv') and all(k.lower() in f_lower for k in keywords):
                return Path(root) / f
    return None

FEATURES_FILE = find_csv_file(['feature']) or find_csv_file(['elliptic_txs_features'])
CLASSES_FILE  = find_csv_file(['class']) or find_csv_file(['elliptic_txs_classes'])
EDGELIST_FILE = find_csv_file(['edge']) or find_csv_file(['elliptic_txs_edgelist'])

print("🔍 Auto-detected CSV Paths:")
print(f"   Features : {FEATURES_FILE}")
print(f"   Classes  : {CLASSES_FILE}")
print(f"   Edgelist : {EDGELIST_FILE}")

if not (FEATURES_FILE and CLASSES_FILE and EDGELIST_FILE):
    print("\n❌ Could not find all 3 Elliptic CSV files under /kaggle/input!")
    print("Files found under /kaggle/input:")
    for r, _, fs in os.walk(base_input):
        for f in fs:
            print(f"   - {os.path.join(r, f)}")
    raise FileNotFoundError("Please attach the Elliptic Bitcoin Dataset in the Kaggle sidebar (+ Add Data).")

print("\n📂 Loading CSVs (this may take ~20-30 seconds)...\n")

# 1. Features: Elliptic features file usually has no headers
sample_line = pd.read_csv(FEATURES_FILE, nrows=1, header=None)
# Check if first column has string header or numeric txId
first_val = str(sample_line.iloc[0, 0]).strip()
if first_val.isdigit() or '.' in first_val:
    features_df = pd.read_csv(FEATURES_FILE, header=None)
else:
    features_df = pd.read_csv(FEATURES_FILE)

# Assign standard column names: txId, time_step, f2, f3, ...
feature_cols_count = features_df.shape[1] - 2
features_df.columns = ['txId', 'time_step'] + [f'f{i}' for i in range(2, 2 + feature_cols_count)]

# 2. Classes
classes_df = pd.read_csv(CLASSES_FILE)
classes_df.columns = [c.strip() for c in classes_df.columns]
if 'txId' not in classes_df.columns:
    classes_df.rename(columns={classes_df.columns[0]: 'txId', classes_df.columns[1]: 'class'}, inplace=True)
classes_df['class'] = classes_df['class'].astype(str).str.strip()

# 3. Edgelist
edges_df = pd.read_csv(EDGELIST_FILE)
edges_df.columns = [c.strip() for c in edges_df.columns]
if 'txId1' not in edges_df.columns or 'txId2' not in edges_df.columns:
    edges_df.rename(columns={edges_df.columns[0]: 'txId1', edges_df.columns[1]: 'txId2'}, inplace=True)

# Merge features and labels
df = features_df.merge(classes_df, on='txId', how='left')
df['class'] = df['class'].fillna('unknown')

# Elliptic convention: '1' = illicit (fraud), '2' = licit (regular), 'unknown' = unlabelled (-1)
label_map = {'1': 1, '2': 0, 'unknown': -1}
df['label'] = df['class'].map(label_map).fillna(-1).astype(int)

# Temporal Splits (Steps 1-34 Train, 35-41 Val, 42-49 Test)
train_idx = df['time_step'].isin(range(1, 35))
val_idx   = df['time_step'].isin(range(35, 42))
test_idx  = df['time_step'].isin(range(42, 50))

print(f"📊 Total Transactions: {len(df):,}")
print(f"   - Illicit (1)      : {(df['label'] == 1).sum():,}")
print(f"   - Licit (0)        : {(df['label'] == 0).sum():,}")
print(f"   - Unlabelled (-1)  : {(df['label'] == -1).sum():,}")
print(f"✂️ Splits:")
print(f"   - Train Nodes      : {train_idx.sum():,}")
print(f"   - Val Nodes        : {val_idx.sum():,}")
print(f"   - Test Nodes       : {test_idx.sum():,}")
print(f"🔗 Total Edges        : {len(edges_df):,}")


In [ ]:
# ============================================================
# CELL 3 — Build PyTorch Geometric Data Object
# ============================================================
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler

print("🔨 Assembling PyTorch Geometric Graph...")

# 1. Feature normalization (fit ONLY on training split to prevent leakage)
feature_cols = [c for c in df.columns if c.startswith('f')]
scaler = StandardScaler()
X_raw = df[feature_cols].values.astype(np.float32)
scaler.fit(X_raw[train_idx.values])
X_norm = scaler.transform(X_raw)

X = torch.tensor(X_norm, dtype=torch.float)
y = torch.tensor(df['label'].values, dtype=torch.long)

# 2. Split masks
train_mask = torch.tensor(train_idx.values, dtype=torch.bool)
val_mask   = torch.tensor(val_idx.values, dtype=torch.bool)
test_mask  = torch.tensor(test_idx.values, dtype=torch.bool)

# 3. Fast Edge Index construction via dictionary lookup
print("   Indexing edges...")
id_to_idx = {txid: i for i, txid in enumerate(df['txId'].values)}

valid_edges = edges_df[edges_df['txId1'].isin(id_to_idx) & edges_df['txId2'].isin(id_to_idx)]
src = [id_to_idx[tx] for tx in valid_edges['txId1'].values]
dst = [id_to_idx[tx] for tx in valid_edges['txId2'].values]
edge_index = torch.tensor([src, dst], dtype=torch.long)

# 4. PyG Data object
graph_data = Data(
    x=X,
    edge_index=edge_index,
    y=y,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask,
)

print(f"\n✅ Graph constructed successfully:")
print(f"   Nodes     : {graph_data.num_nodes:,}")
print(f"   Edges     : {graph_data.edge_index.shape[1]:,}")
print(f"   Features  : {graph_data.x.shape[1]}")


In [ ]:
# ============================================================
# CELL 4 — Define the Graph Transformer Model Architecture
# ============================================================
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import TransformerConv

class GraphTransformerClassifier(nn.Module):
    """
    3-layer Graph Transformer for node-level fraud detection.
    Uses multi-head attention (TransformerConv) to attend to 1-hop and multi-hop neighbors.
    Logs attention weights on demand for Stage 6 explainability.
    """
    def __init__(
        self,
        in_channels: int,
        hidden_channels: int = 128,
        num_classes: int = 2,
        heads: int = 4,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.dropout = dropout
        self.heads   = heads

        head_dim = hidden_channels // heads

        self.conv1 = TransformerConv(in_channels, head_dim, heads=heads,
                                     dropout=dropout, concat=True, beta=True)
        self.bn1   = nn.BatchNorm1d(hidden_channels)

        self.conv2 = TransformerConv(hidden_channels, head_dim, heads=heads,
                                     dropout=dropout, concat=True, beta=True)
        self.bn2   = nn.BatchNorm1d(hidden_channels)

        self.conv3 = TransformerConv(hidden_channels, head_dim, heads=heads,
                                     dropout=dropout, concat=True, beta=True)
        self.bn3   = nn.BatchNorm1d(hidden_channels)

        self.classifier = nn.Linear(hidden_channels, num_classes)
        self.attention_weights = {}

    def forward(self, x, edge_index, return_attention_weights: bool = False):
        # Layer 1
        if return_attention_weights:
            x, (edge_idx1, attn1) = self.conv1(x, edge_index, return_attention_weights=True)
            self.attention_weights['layer1'] = (edge_idx1, attn1.detach())
        else:
            x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # Layer 2
        if return_attention_weights:
            x, (edge_idx2, attn2) = self.conv2(x, edge_index, return_attention_weights=True)
            self.attention_weights['layer2'] = (edge_idx2, attn2.detach())
        else:
            x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # Layer 3
        if return_attention_weights:
            x, (edge_idx3, attn3) = self.conv3(x, edge_index, return_attention_weights=True)
            self.attention_weights['layer3'] = (edge_idx3, attn3.detach())
        else:
            x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        return self.classifier(x)

# Instantiate model
IN_DIM  = graph_data.x.shape[1]
HIDDEN  = 128
HEADS   = 4
DROPOUT = 0.3

model = GraphTransformerClassifier(
    in_channels     = IN_DIM,
    hidden_channels = HIDDEN,
    heads           = HEADS,
    dropout         = DROPOUT,
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ GraphTransformerClassifier instantiated:")
print(f"   Input dim  : {IN_DIM}")
print(f"   Hidden dim : {HIDDEN} ({HEADS} heads x {HIDDEN // HEADS})")
print(f"   Parameters : {n_params:,}")


In [ ]:
# ============================================================
# CELL 5 — Mini-batch NeighborLoader Training on GPU
# ============================================================
import time
from sklearn.metrics import f1_score, roc_auc_score
from torch_geometric.loader import NeighborLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Running on: {device}")

graph_data = graph_data.to(device)
model      = model.to(device)

# Compute inverse class weights to counteract fraud rarity
train_labeled = graph_data.y[graph_data.train_mask & (graph_data.y >= 0)]
n_total   = float(len(train_labeled))
n_licit   = float((train_labeled == 0).sum().item())
n_illicit = float((train_labeled == 1).sum().item())
class_wts = torch.tensor([n_total / (2.0 * n_licit), n_total / (2.0 * n_illicit)], dtype=torch.float, device=device)
print(f"⚖️ Loss class weights: licit={class_wts[0]:.3f}, illicit={class_wts[1]:.3f}")

# Mini-batch NeighborLoader for scalable neighborhood sampling
train_loader = NeighborLoader(
    graph_data,
    num_neighbors = [25, 10, 5],
    batch_size    = 512,
    input_nodes   = graph_data.train_mask & (graph_data.y >= 0),
    shuffle       = True,
)

criterion = nn.CrossEntropyLoss(weight=class_wts)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60)

# Evaluation function (threshold sweep for minority class F1)
@torch.no_grad()
def evaluate(data, mask):
    model.eval()
    logits = model(data.x, data.edge_index)
    eval_mask = mask & (data.y >= 0)
    probs  = F.softmax(logits[eval_mask], dim=-1)[:, 1].cpu().numpy()
    labels = data.y[eval_mask].cpu().numpy()

    best_f1 = 0.0
    for thresh in np.arange(0.05, 0.96, 0.05):
        preds = (probs >= thresh).astype(int)
        f1 = f1_score(labels, preds, pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1

    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = 0.5

    return best_f1, auc

# Training Loop
NUM_EPOCHS = 60
PATIENCE   = 12
best_val_f1 = 0.0
best_epoch  = 0
no_improve  = 0
history = {'train_loss': [], 'val_f1': [], 'val_auc': []}
CKPT_PATH = '/kaggle/working/graph_transformer.pt'

print(f"\n🚀 Training Graph Transformer for up to {NUM_EPOCHS} epochs...\n")
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val F1':>8} | {'Val AUC':>8} | {'LR':>9} | {'Time':>6}")
print('-' * 62)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    t0 = time.time()
    total_loss = 0.0
    n_batches = 0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        logits = model(batch.x, batch.edge_index)
        seed_n = batch.batch_size
        seed_logits = logits[:seed_n]
        seed_labels = batch.y[:seed_n]

        labelled_mask = seed_labels >= 0
        if labelled_mask.sum() == 0:
            continue

        loss = criterion(seed_logits[labelled_mask], seed_labels[labelled_mask])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    scheduler.step()
    avg_loss = total_loss / max(n_batches, 1)
    val_f1, val_auc = evaluate(graph_data, graph_data.val_mask)
    elapsed = time.time() - t0
    lr_now = optimizer.param_groups[0]['lr']

    history['train_loss'].append(avg_loss)
    history['val_f1'].append(val_f1)
    history['val_auc'].append(val_auc)

    star = '  ⭐ best' if val_f1 > best_val_f1 else ''
    print(f"{epoch:>6} | {avg_loss:>10.4f} | {val_f1:>8.4f} | {val_auc:>8.4f} | {lr_now:>9.6f} | {elapsed:>5.1f}s{star}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch  = epoch
        no_improve  = 0
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'val_f1': val_f1,
            'val_auc': val_auc,
            'in_channels': IN_DIM,
            'hidden_dim': HIDDEN,
            'heads': HEADS,
            'dropout': DROPOUT,
        }, CKPT_PATH)
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f"\n⏹️ Early stopping at epoch {epoch} (no improvement in {PATIENCE} epochs).")
        break

print(f"\n🏆 Best Validation F1: {best_val_f1:.4f} (Epoch {best_epoch})")


In [ ]:
# ============================================================
# CELL 6 — Final Test Split Evaluation
# ============================================================
# Load best checkpoint
if Path(CKPT_PATH).exists():
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    print(f"Loaded best model from Epoch {ckpt['epoch']}.")

test_f1, test_auc = evaluate(graph_data, graph_data.test_mask)

print('\n' + '=' * 50)
print('📊 STAGE 5 — GRAPH TRANSFORMER FINAL TEST RESULTS')
print('=' * 50)
print(f'  Best Val F1 : {best_val_f1:.4f}')
print(f'  Test F1     : {test_f1:.4f}')
print(f'  Test AUC    : {test_auc:.4f}')
print('=' * 50)

BASELINE_VAL_F1 = 0.85
if best_val_f1 >= BASELINE_VAL_F1:
    print(f"\n✅ [PASS] Transformer Val F1 ({best_val_f1:.4f}) meets target ({BASELINE_VAL_F1})!")
else:
    print(f"\nℹ️ Transformer Val F1 is {best_val_f1:.4f}.")


In [ ]:
# ============================================================
# CELL 7 — Plot & Save Learning Curves
# ============================================================
import matplotlib.pyplot as plt

epochs = range(1, len(history['val_f1']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs, history['train_loss'], color='#2563eb', lw=2)
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['val_f1'], color='#16a34a', lw=2)
axes[1].set_title('Validation F1 (Illicit)', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, history['val_auc'], color='#9333ea', lw=2)
axes[2].set_title('Validation ROC-AUC', fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()
print("📈 Plot saved to /kaggle/working/training_curves.png")


In [ ]:
# ============================================================
# CELL 8 — Verify Generated Output Files
# ============================================================
working_dir = Path('/kaggle/working')
print("📦 Files generated in /kaggle/working/:")
print('-' * 45)
for f in sorted(working_dir.iterdir()):
    if f.is_file():
        mb = f.stat().st_size / (1024 * 1024)
        print(f"  • {f.name:<30} {mb:>6.2f} MB")
print('-' * 45)
print("\n📥 NEXT STEP:")
print("1. In the right panel of Kaggle, open the 'Output' section.")
print("2. Download 'graph_transformer.pt'.")
print("3. Place it into your project folder at: models/graph_transformer.pt")
print("4. You can now proceed to Stage 6 (Explainability with GNNExplainer)!")
